# Download mmCIF files and build protein contact graphs

Build chain-level C-alpha contact graphs for the active manifest selected in notebook 01. Cached structures and compatible graphs are reused, graph caches are separated by contact-distance settings, and missing mmCIFs download concurrently, and protein graphs are built concurrently with configurable worker pools. Separate progress bars track both phases.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "notebooks" / "src" / "enzyme_workflow").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the repository root from this notebook's working directory")
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))
from enzyme_workflow.dataset import build_graph_dataset


In [2]:
# User parameters
MIN_CONTACT_DISTANCE = 4.0
MAX_CONTACT_DISTANCE = 8.0
OVERWRITE = False
SAMPLE_LIMIT = None  # Set to a small integer for a quick first run.
DOWNLOAD_TIMEOUT_SECONDS = 60
DOWNLOAD_MAX_RETRIES = 4
DOWNLOAD_RETRY_BACKOFF_SECONDS = 2.0
N_JOBS = 8  # Parallel protein graph construction workers.
DOWNLOAD_JOBS = 8  # Parallel downloads for distinct PDB entries.


In [3]:
build_result = build_graph_dataset(
    PROJECT_ROOT,
    min_contact_distance=MIN_CONTACT_DISTANCE,
    max_contact_distance=MAX_CONTACT_DISTANCE,
    overwrite=OVERWRITE,
    sample_limit=SAMPLE_LIMIT,
    download_timeout_seconds=DOWNLOAD_TIMEOUT_SECONDS,
    download_max_retries=DOWNLOAD_MAX_RETRIES,
    download_retry_backoff_seconds=DOWNLOAD_RETRY_BACKOFF_SECONDS,
    n_jobs=N_JOBS,
    download_jobs=DOWNLOAD_JOBS,
)
display(build_result.class_counts)
display(build_result.status_counts)
display(build_result.status.head())


Manifest source: ./data/interim/ec_protein_manifest_level_2.tsv (EC level 2)
Manifest samples: 411
Existing graph caches: 136/411
Existing mmCIF files: 138/409 unique PDB entries
Graph cache: ./data/processed/graphs/contact_4_8A (contact range 4–8 Å)
Parallel workers: 8 graph workers; 8 structure-download workers


Building contact graphs:   0%|          | 0/411 [00:00<?, ?protein/s]

,samples
ec_number,
1.1,10
1.10,1
1.11,10
1.13,10
1.14,10
1.15,2
1.16,7
1.17,5
1.18,1


,proteins
status,
created,275
skipped_existing,136


,sample_id,pdb_id,label_asym_id,auth_asym_id,graph_path,status,error,updated_at_utc
0,8u7q.H,8u7q,H,H,.,skipped_existing,,2026-11-18T11:40:02+00:00
1,8zkj.B,8zkj,B,B,.,skipped_existing,,2026-11-18T11:40:02+00:00
2,6ua5.H,6ua5,H,H,.,skipped_existing,,2026-11-18T11:40:02+00:00
3,1yb1.B,1yb1,B,B,.,skipped_existing,,2026-11-18T11:40:02+00:00
4,6jkh.B,6jkh,B,B,.,skipped_existing,,2026-11-18T11:40:02+00:00


Each raw graph is an undirected NetworkX graph with one node per C-alpha residue and edges for contacts in the configured inclusive distance window. Notebook 03 derives vectorizer-ready labels without changing the cached graph.
